# LSTM from scratch

This notebook implements an LSTM with basic PyTorch operations, checks it against `nn.LSTM`, and applies it to a visual task: classifying MNIST digits by reading each image one row at a time.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/07_rnn'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

## LSTM cell

In a vanilla RNN, the hidden state is rewritten at every step. An LSTM adds a cell state $c_t$ that is changed only through gates, which are values between 0 and 1:

$$i_t = \sigma(W_{xi} x_t + W_{hi} h_{t-1} + b_i), \qquad f_t = \sigma(W_{xf} x_t + W_{hf} h_{t-1} + b_f)$$

$$g_t = \tanh(W_{xg} x_t + W_{hg} h_{t-1} + b_g), \qquad o_t = \sigma(W_{xo} x_t + W_{ho} h_{t-1} + b_o)$$

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)$$

The forget gate $f_t$ decides what to keep from the cell, the input gate $i_t$ decides how much of the candidate $g_t$ to write, and the output gate $o_t$ decides what to expose in $h_t$. Since $c_t$ is updated by an element-wise product and a sum, gradients can flow through many steps with less decay than in a vanilla RNN.

The four linear maps have the same input, so they are computed with a single matrix of size $4H$ and split afterwards.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)


class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        bound = hidden_size ** -0.5
        self.W_x = nn.Parameter(torch.empty(input_size, 4 * hidden_size).uniform_(-bound, bound))
        self.W_h = nn.Parameter(torch.empty(hidden_size, 4 * hidden_size).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.zeros(4 * hidden_size))
        self.b.data[hidden_size:2 * hidden_size] = 1  # the forget gate starts open, so the cell keeps its content

    def forward(self, x, state):
        """
        Inputs:
        - x: Input of the current step, of shape (N, D)
        - state: Tuple (h, c) with the previous states, each of shape (N, H)

        Returns the tuple (h, c) with the next states.
        """
        h, c = state
        gates = x @ self.W_x + h @ self.W_h + self.b  # (N, 4H)
        i, f, g, o = gates.chunk(4, dim=1)  # four tensors of shape (N, H)
        i, f, g, o = torch.sigmoid(i), torch.sigmoid(f), torch.tanh(g), torch.sigmoid(o)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(self, x):
        """
        Inputs:
        - x: Input sequence, of shape (N, T, D)

        Returns the hidden states of all steps, of shape (N, T, H).
        """
        h = x.new_zeros(x.size(0), self.hidden_size)
        c = x.new_zeros(x.size(0), self.hidden_size)
        states = []
        for t in range(x.size(1)):
            h, c = self.cell(x[:, t], (h, c))
            states.append(h)
        return torch.stack(states, dim=1)

## Checking against `nn.LSTM`

If the implementation is correct, it must produce the same output as `nn.LSTM` when both have the same weights. PyTorch uses the same gate order (i, f, g, o), but stores the matrices transposed and has two bias vectors, which are simply added.

In [ ]:
reference = nn.LSTM(input_size=5, hidden_size=7, batch_first=True)
ours = LSTM(input_size=5, hidden_size=7)

with torch.no_grad():
    ours.cell.W_x.copy_(reference.weight_ih_l0.T)
    ours.cell.W_h.copy_(reference.weight_hh_l0.T)
    ours.cell.b.copy_(reference.bias_ih_l0 + reference.bias_hh_l0)

    x = torch.randn(3, 11, 5)
    expected, _ = reference(x)
    difference = (ours(x) - expected).abs().max().item()

print(f"Maximum difference: {difference:.2e}")

## Application: reading digits row by row

An image can also be seen as a sequence. Here, each 28x28 MNIST image is a sequence of 28 steps, and each step is one row of 28 pixels. The network never sees the whole image at once: everything it knows about the rows above must be stored in its state.

The loss is applied after every row, not only after the last one. This asks the network for its best guess at each moment, which we will use at the end of the notebook. The final prediction is the one made after the last row.

The dataset is stored in the disk of the Colab VM, not in your Drive.

In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST("/tmp/mnist", train=True, download=True, transform=transform)
test_set = datasets.MNIST("/tmp/mnist", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512)


class RowClassifier(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        self.lstm = LSTM(28, hidden_size)
        self.output = nn.Linear(hidden_size, 10)

    def forward(self, images):
        rows = images.squeeze(1)  # (N, 1, 28, 28) -> (N, 28, 28): 28 steps with 28 values each
        return self.output(self.lstm(rows))  # logits after each row: (N, 28, 10)


model = RowClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_history = []
for epoch in range(3):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)  # (N, 28, 10)
        targets = labels.unsqueeze(1).expand(-1, 28)  # (N,) -> (N, 28): the same label after each row
        loss = F.cross_entropy(logits.reshape(-1, 10), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

plt.plot(loss_history)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss History")
plt.show()

In [ ]:
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images)[:, -1].argmax(dim=1) == labels).sum().item()
print(f"Test accuracy: {correct / len(test_set):.4f}")

## How the prediction evolves

The curves show the probability of the correct digit after each row. In the first rows the image is still black, so the network can only guess. The probability then changes as the strokes appear, and it can also fall when the visible rows look like another digit.

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    probabilities = F.softmax(model(images[:5].to(device)), dim=2).cpu()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(images[i, 0], cmap="gray")
    axes[0, i].set_title(f"Digit {labels[i].item()}")
    axes[0, i].axis("off")
    axes[1, i].plot(probabilities[i, :, labels[i]])
    axes[1, i].set_ylim(0, 1.05)
    axes[1, i].set_xlabel("Rows read")
axes[1, 0].set_ylabel("Probability of the correct digit")
plt.tight_layout()
plt.show()